# Pytest: Testing Engineering Code

## The Core Question

**How do you verify your simulation code is correct before trusting its results?**

You wrote a drag-coefficient function. It *looks* right. But is the sign correct? Is the unit conversion in the right place? Does it blow up at Re = 0? You won't find out until you've run a 1000-hour simulation and the results are garbage.

Pytest lets you catch those bugs in **seconds**, not days.

## Why this library?

Pytest is the de-facto standard Python testing framework. Unlike `unittest`, it needs no boilerplate — a plain `assert` is a test. It discovers tests automatically, shares setup via **fixtures**, and reports failures with the exact value that went wrong. For an engineer, that means: *test the physics against known values before you trust the physics.*

## The Workflow

```
  write physics function
        |
        v
  write tests (assert known values)
        |
        v
  run:  pytest -v
        |
        +--> all green  -->  trust the code, run the big sim
        +--> red         -->  fix the bug NOW (cheap), not after 1000 h
```


In [ ]:
import pytest
print('pytest', pytest.__version__)

## 1. The Physics Function Under Test

We test a small, realistic function: the **drag coefficient of a smooth sphere** as a function of Reynolds number. It uses the standard empirical regimes:

- **Stokes** (Re < 1): `Cd = 24/Re + 6/(1+sqrt(Re))`
- **Schiller-Naumann** (1 < Re < 1000): `Cd = 24/Re (1 + 0.15 Re^0.687)`
- **Subcritical** (1000 < Re < 2e5): `Cd = 0.47` (the classic textbook value)
- **Supercritical** (Re > 2e5): `Cd = 0.2`

The function also **guards** against unphysical input (Re <= 0), which is exactly the kind of bug a test should catch.


In [ ]:
import math

def drag_coefficient(Re: float) -> float:
    """Drag coefficient of a smooth sphere vs. Reynolds number."""
    if Re <= 0:
        raise ValueError("Reynolds number must be positive")
    if Re < 1.0:
        return 24.0 / Re + 6.0 / (1.0 + math.sqrt(Re))
    if Re < 1000.0:
        return 24.0 / Re * (1.0 + 0.15 * Re**0.687)
    if Re < 2e5:
        return 0.47
    return 0.2

# A quick sanity print (not a test)
print('Cd at Re=1e5 :', drag_coefficient(1e5))
print('Cd at Re=1e6 :', drag_coefficient(1e6))

## 2. Write the Tests

A test is just a function named `test_*` containing `assert` statements. We write four:

1. **Known value** — the textbook anchor: `Cd = 0.47` at `Re = 1e5`.
2. **Parametrized** — one test, many reference cases (Stokes, transition, subcritical, supercritical).
3. **Edge case** — `Re = 0` must raise, not silently return `inf`.
4. **Edge case** — negative `Re` must raise.

We also define a **fixture** (`fluid`) that hands shared physical constants to any test that asks for it — no copy-paste of `rho` and `mu`.


In [ ]:
import pytest
from sphere_drag import drag_coefficient

# Module-level reference data (used by parametrize)
REFERENCE_CASES = [
    (1e-3, 24000.0, 1e-3),   # Stokes regime
    (1.0, 27.6, 1e-2),       # Schiller-Naumann regime
    (1e5, 0.47, 1e-6),       # subcritical (classic value)
    (1e6, 0.2, 1e-6),        # supercritical
]

@pytest.fixture
def fluid():
    """Shared test data: air at 20 C, 1 atm (density, dynamic viscosity)."""
    return {"rho": 1.204, "mu": 1.825e-5}

def test_known_value_subcritical():
    """Cd = 0.47 for a sphere at Re = 1e5 (textbook value)."""
    assert drag_coefficient(1e5) == pytest.approx(0.47, abs=1e-6)

@pytest.mark.parametrize("Re, expected, tol", REFERENCE_CASES)
def test_correlation_matches_reference(Re, expected, tol):
    assert drag_coefficient(Re) == pytest.approx(expected, rel=tol)

def test_zero_reynolds_raises():
    with pytest.raises(ValueError):
        drag_coefficient(0.0)

def test_negative_reynolds_raises():
    with pytest.raises(ValueError):
        drag_coefficient(-5.0)

def test_fixture_is_available(fluid):
    """The fixture provides shared physical constants to any test."""
    assert fluid["rho"] > 0
    assert fluid["mu"] > 0

### What does this mean?

- `pytest.approx` is the right way to compare **floats** in engineering code — never `==`. It lets you set a tolerance (`abs` or `rel`) instead of fighting floating-point noise.
- `@pytest.mark.parametrize` turns one test into four, so a single regression in *any* regime fails the suite.
- `pytest.raises` asserts that **invalid input is rejected** — the boundary conditions that crash a solver at 3 a.m.

**Engineering takeaway:** the tests encode *known physics*. If someone later "refactors" the function and flips a sign or drops a unit conversion, the suite goes red immediately — long before a production run.


## 3. Run the Suite

Pytest discovers every `test_*` function automatically. Run it with `-v` for a per-test report.


In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

### Reading the output

```
test_sphere_drag.py::test_known_value_subcritical PASSED
test_sphere_drag.py::test_correlation_matches_reference[0.001-24000.0-0.001] PASSED
test_sphere_drag.py::test_correlation_matches_reference[1.0-27.6-0.01] PASSED
test_sphere_drag.py::test_correlation_matches_reference[100000.0-0.47-1e-06] PASSED
test_sphere_drag.py::test_correlation_matches_reference[1000000.0-0.2-1e-06] PASSED
test_sphere_drag.py::test_zero_reynolds_raises PASSED
test_sphere_drag.py::test_negative_reynolds_raises PASSED
test_sphere_drag.py::test_fixture_is_available PASSED
============================== 8 passed in 0.02s ==============================
```

Eight tests, all green, in 0.02 s. That is the entire cost of *knowing* the drag function is correct.

**Engineering takeaway:** a green suite is a **certificate of correctness** for the parts you tested. The more of your physics you pin to known values, the more you can trust the rest of the pipeline.


## 4. Exercises

1. **Add a regime.** Extend `drag_coefficient` with the supercritical *drag crisis* (a dip near Re = 3e5). Add a parametrized test for it.

2. **Tighten a tolerance.** Change the subcritical test to `abs=1e-9`. Does it still pass? What does that tell you about the function's precision?

3. **Break it on purpose.** Change `0.47` to `0.74` in the function. Run `pytest -v`. Read the failure message — it shows the *obtained* vs *expected* value. This is the exact feedback you want before a big run.

4. **Fixture in action.** Write a test that uses the `fluid` fixture to compute the Reynolds number of a 10 mm sphere at 10 m/s in air, and assert it is in the subcritical regime.
